# 05 · Figuras con R base

Ahora convertiremos las tablas de la clase 04 en figuras. Ejecuta en orden y modifica un argumento a la vez para observar su efecto.

## Objetivos y recorrido

- Elegir una figura según el tipo de variable y la pregunta.
- Dibujar puntos, líneas y barras con funciones de R base.
- Explorar distribuciones con histogramas, densidades, boxplots y gráficos Q–Q.
- Añadir títulos, unidades, grupos y leyendas, y guardar una figura.

Una figura necesita una pregunta, variables bien definidas y una explicación. La forma más vistosa no siempre comunica mejor.


## Recuperar las tablas del notebook 04

Este notebook es independiente: vuelve a leer los tres CSV. Desde `estudiantes/clases/` usa `../../data/raw`; desde la raíz del repositorio cambia la carpeta a `data/raw`. En Colab carga los tres archivos y usa `/content`. Antes de continuar, las tres comprobaciones deben dar `TRUE`.

Conservamos el filtro de Colombia por `place`, los metadatos y ausentes de NASA, y las fechas calculadas con año y DOY. Las fuentes y sus enlaces están en [la clase 04](04_lectura_y_analisis_exploratorio.ipynb).


In [ ]:
carpeta_datos <- "../../data/raw"
archivo_sismos <- file.path(carpeta_datos, "Earthquakes in Colombia (from 2000).csv")
archivo_meteo <- file.path(carpeta_datos, "Datos hidrometeorológicos Medellín.csv")
archivo_pinguinos <- file.path(carpeta_datos, "DB Pinguinos.csv")
file.exists(c(archivo_sismos, archivo_meteo, archivo_pinguinos))


In [ ]:
sismos <- read.csv(archivo_sismos, fileEncoding = "UTF-8",
                   na.strings = c("", "NA"), stringsAsFactors = FALSE)
sismos <- sismos[!is.na(sismos$place) & grepl("Colombia", sismos$place, fixed = TRUE), ]
meteo <- read.csv(archivo_meteo, skip = 12, fileEncoding = "UTF-8",
                  na.strings = c("", "NA", "-999"), stringsAsFactors = FALSE)
meteo$fecha <- as.Date(paste0(meteo$YEAR, "-01-01")) + meteo$DOY - 1
meteo$mes <- as.integer(format(meteo$fecha, "%m"))
meteo <- meteo[order(meteo$fecha), ]
pinguinos <- read.csv(archivo_pinguinos, fileEncoding = "UTF-8",
                      na.strings = c("", "NA"), stringsAsFactors = FALSE)
print(c(sismos = nrow(sismos), dias = nrow(meteo), pinguinos = nrow(pinguinos)))


### Conservar los datos que necesita cada figura

`complete.cases()` identifica filas conocidas en las columnas seleccionadas. Preparamos una tabla para la relación profundidad–magnitud y otra para al menos dos medidas corporales y la especie. No eliminamos una fila por ausentes en variables ajenas a la pregunta. Las tablas originales permanecen disponibles.


In [ ]:
pares_sismos <- sismos[complete.cases(sismos[, c("depth", "mag")]), ]
pares_bio <- pinguinos[complete.cases(pinguinos[, c("species", "flipper_length_mm", "body_mass_g")]), ]
print(c(pares_sismos = nrow(pares_sismos), pares_bio = nrow(pares_bio)))


## Elegir el tipo de figura

| Pregunta | Variables | Figura |
|---|---|---|
| ¿Cómo se relacionan dos medidas? | Dos numéricas | Dispersión |
| ¿Cómo varía una medida en el tiempo? | Fecha y medida | Líneas o puntos ordenados |
| ¿Cuántos registros hay en cada categoría? | Una categórica | Barras |
| ¿Cómo se distribuye una medida? | Una numérica | Histograma o densidad |
| ¿Cómo cambia una distribución por grupo? | Numérica y categórica | Boxplot |
| ¿Se aproxima una muestra a una distribución normal? | Una numérica | Q–Q normal |

“Gráficos clásicos” se refiere aquí a las funciones de R base. La distribución normal es una referencia estadística específica que veremos al final.


## Estructura básica de una dispersión

```r
plot(x, y,
     main = "Título",
     xlab = "Variable X (unidad)", ylab = "Variable Y (unidad)",
     pch = 16, col = "steelblue")
```

`x` controla la posición horizontal; `y`, la vertical. `pch` cambia el símbolo, `col` su color y `cex` su tamaño. Comenzamos con las mismas variables sísmicas de la clase 04.


In [ ]:
plot(pares_sismos$depth, pares_sismos$mag,
     main = "Sismos con Colombia en su localización",
     xlab = "Profundidad (km)", ylab = "Magnitud",
     pch = 16, cex = 0.6, col = adjustcolor("steelblue", alpha.f = 0.35))


Cada punto es un evento. La transparencia ayuda donde se superponen. Una asociación visual no prueba causalidad; además, el catálogo mezcla tipos de magnitud (`magType`) y su selección no representa todos los eventos ocurridos en el país.

### Añadir grupos y una leyenda

En pingüinos, un color por especie ayuda a distinguir relaciones dentro y entre grupos. La leyenda explica los colores. `points()` agrega puntos a una figura existente; `plot()` inicia una nueva.


In [ ]:
plot(pares_bio$flipper_length_mm, pares_bio$body_mass_g,
     pch = 16, col = 'red', cex = 0.6,
     xlab = "Longitud de la aleta (mm)", ylab = "Masa corporal (g)",
     main = "Medidas corporales por especie")

In [ ]:
colores_especies <- c(Adelie = "darkorange3", Chinstrap = "purple4", Gentoo = "steelblue4")
plot(pares_bio$flipper_length_mm, pares_bio$body_mass_g,
     pch = 16, col = colores_especies[pares_bio$species],
     xlab = "Longitud de la aleta (mm)", ylab = "Masa corporal (g)",
     main = "Medidas corporales por especie")
legend("topleft", legend = names(colores_especies),
       col = colores_especies, pch = 16, bty = "n")


## Líneas: ordenar antes de unir

`type = "l"` une puntos en el orden de la tabla. Eso tiene sentido para fechas ordenadas. Compararemos enero entre todos los años, como en la clase 04: cada punto será un promedio mensual, no un registro diario.

Consultamos los días conocidos para que una falta de datos no se confunda con un cambio de temperatura. Enero está incluido en todos los años de este archivo.


In [ ]:
enero <- meteo[meteo$mes == 1 & !is.na(meteo$T2M), ]
table(enero$YEAR)
temperatura_enero <- aggregate(T2M ~ YEAR, data = enero, FUN = mean)
temperatura_enero <- temperatura_enero[order(temperatura_enero$YEAR), ]
head(temperatura_enero)


In [ ]:
plot(temperatura_enero$YEAR, temperatura_enero$T2M,
     type = "b", pch = 16, col = "tomato3",
     main = "Temperatura media de enero", xlab = "Año", ylab = "Temperatura a 2 m (°C)")
abline(h = mean(temperatura_enero$T2M), lty = 2, col = "grey40")


`type = "b"` muestra puntos y líneas. `abline(h = ...)` añade una referencia horizontal. Aquí representa la media de los promedios de enero; no es una estimación de tendencia climática.

### Precipitación diaria

Para una cantidad de cada día podemos usar segmentos (`type = "h"`). La selección de 2025 conserva un año completo. Los datos NASA POWER corresponden a una celda del producto, no a una estación meteorológica puntual.


In [ ]:
meteo_2025 <- meteo[meteo$YEAR == 2025, ]
plot(meteo_2025$fecha, meteo_2025$PRECTOTCORR, type = "h", col = "steelblue",
     main = "Precipitación diaria · 2025", xlab = "Fecha", ylab = "Precipitación (mm/día)")


## Barras para categorías

`table()` cuenta los registros por especie y `barplot()` representa esos conteos. Una barra corresponde a una categoría, no a un intervalo numérico. Los conteos describen esta muestra; no estiman la abundancia poblacional de pingüinos.

```r
conteos <- table(tabla$categoria)
barplot(conteos, xlab = "Categoría", ylab = "Número de registros")
```


In [ ]:
conteos_especies <- table(pinguinos$species)
print(conteos_especies)
barplot(conteos_especies, col = colores_especies[names(conteos_especies)],
        xlab = "Especie", ylab = "Individuos registrados", main = "Composición de la muestra")


## Histogramas: frecuencias por intervalos

`hist()` agrupa una variable numérica en intervalos. `breaks` propone una partición; prueba 10 y 30 para ver cómo cambia la impresión visual sin cambiar los datos.

En precipitación, cero es una observación válida. Los `-999` ya se interpretaron como `NA` al leer el CSV.


In [ ]:
lluvia <- meteo$PRECTOTCORR[is.finite(meteo$PRECTOTCORR)]
hist(lluvia, breaks = 30, col = "lightblue", border = "white",
     main = "Distribución de la precipitación diaria",
     xlab = "Precipitación (mm/día)", ylab = "Número de días")


### Densidad: una representación suavizada

Una densidad tiene área total uno; su altura no es un conteo. Para superponerla, el histograma debe usar `probability = TRUE`. `density()` suaviza y puede dibujar una cola fuera del dominio físico; no lo interpretes como observaciones negativas.

Comparamos el histograma de magnitud con una curva suavizada. `lines()` añade una línea a la figura abierta.


In [ ]:
hist(pares_sismos$mag, breaks = 20, probability = TRUE,
     col = "wheat", border = "white", main = "Magnitud: histograma y densidad",
     xlab = "Magnitud", ylab = "Densidad")
lines(density(pares_sismos$mag), lwd = 2, col = "firebrick")


## Boxplots: comparar distribuciones

```r
boxplot(medida ~ grupo, data = tabla, xlab = "Grupo", ylab = "Medida (unidad)")
```

La línea central es la mediana; la caja abarca del primer al tercer cuartil. Los bigotes alcanzan observaciones dentro de 1.5 veces el rango intercuartílico desde la caja. Los puntos restantes se señalan como potencialmente atípicos, pero no se eliminan automáticamente.

La fórmula `body_mass_g ~ species` indica comparar la masa entre especies. La forma de cada grupo importa tanto como su centro.


In [ ]:
boxplot(body_mass_g ~ species, data = pares_bio,
        col = colores_especies[sort(unique(pares_bio$species))],
        main = "Masa corporal por especie", xlab = "Especie", ylab = "Masa (g)")


## Varias medidas: matriz de dispersiones

`pairs()` compara cada par de columnas numéricas. La diagonal identifica la variable. No mezcles categorías codificadas como enteros con medidas físicas: un código de especie no se convierte por ello en una cantidad.


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 7)
medidas <- pinguinos[, c("bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g")]
filtro_filas_completas <- complete.cases(medidas)
medidas <- medidas[filtro_filas_completas, ]
colores_puntos <- colores_especies[pinguinos$species[filtro_filas_completas]]
pairs(medidas, pch = 16, cex = 0.35, col = adjustcolor(colores_puntos, alpha.f = 0.4),
      main = "Relaciones entre medidas corporales")


## Dos figuras en una misma ventana

`par(mfrow = c(filas, columnas))` organiza paneles. Guarda los parámetros anteriores y restáuralos para que las siguientes figuras vuelvan a su disposición original.


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
parametros_previos <- par(mfrow = c(1, 2))
hist(pares_sismos$depth, breaks = 20, col = "lightblue",
     main = "Profundidad", xlab = "Profundidad (km)")
boxplot(pares_sismos$depth, col = "lightblue", main = "Resumen de profundidad",
        ylab = "Profundidad (km)")
par(parametros_previos)


## Guardar una figura

El dispositivo `png()` abre un archivo. Las instrucciones gráficas siguientes se dibujan allí y `dev.off()` termina la escritura. La carpeta se crea si no existe; al repetir la celda se actualiza este PNG.

Para una salida vectorial puedes usar `pdf("figuras/masa_por_especie.pdf")` antes de dibujar y cerrar después con `dev.off()`.


In [ ]:
dir.create("figuras", showWarnings = FALSE)
png("figuras/masa_por_especie_base.png", width = 1800, height = 1200, res = 200)
boxplot(body_mass_g ~ species, data = pares_bio,
        xlab = "Especie", ylab = "Masa (g)", main = "Masa corporal por especie",
        col = "lightblue")
dev.off()


## Documentación para consultar

La ayuda de [plot()](https://stat.ethz.ch/R-manual/R-devel/library/graphics/html/plot.default.html) explica sus argumentos. Desde R puedes consultar `?hist`, `?boxplot` y `?qqnorm`.

